# 02 — Data Cleaning & Column Renaming

**NST DVA Capstone 2 · NASA Planetary Defense**  
**Team:** SectionC_G-09

---

## Objectives

- Apply full rename maps to both datasets (all abbreviated JPL names → descriptive names)
- Perform null imputation, type casting, and deduplication
- Derive analysis-ready columns (`risk_tier`, `speed_category`, `orbit_class_label`, etc.)
- Produce and save **4 separate analysis-ready datasets** to `data/processed/`

## Output Datasets

| File | Description |
|---|---|
| `nea_catalogue_clean.csv` | Full cleaned NEA catalogue (all asteroids) |
| `nea_hazardous_clean.csv` | Potentially Hazardous Asteroids (PHAs) only |
| `close_approaches_clean.csv` | All cleaned close approaches (2015–2035) |
| `close_approaches_future_clean.csv` | Future close approaches (≥2025 only) |

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT  = Path().resolve().parent
RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

import etl_pipeline
import cleaning

print(f'Processed output: {PROCESSED_DIR}')

## 2.1 — Load Raw Data with Rename Maps Applied

In [ ]:
close_raw, nea_raw = etl_pipeline.extract_all(RAW_DIR)

print('After extraction + column rename:')
print(f'  Close Approaches: {close_raw.shape}')
print(f'  NEA Catalogue   : {nea_raw.shape}')

print('\nClose Approaches columns:')
print(list(close_raw.columns))

print('\nNEA Catalogue columns (renamed):')
print(list(nea_raw.columns))

## 2.2 — Before / After Column Name Comparison (NEA)

In [ ]:
print(f"{'Original (abbreviated)':<30} → {'New (descriptive)':<40}")
print('─' * 72)
for orig, new in etl_pipeline.NEA_RENAME_MAP.items():
    status = '✓' if new in nea_raw.columns else '–'
    print(f'  {status} {orig:<28} → {new}')

## 2.3 — Apply Full Cleaning Pipeline

In [ ]:
close_clean, nea_clean = cleaning.clean_all(close_raw, nea_raw)

print('After cleaning:')
print(f'  Close Approaches: {close_clean.shape}')
print(f'  NEA Catalogue   : {nea_clean.shape}')

## 2.4 — Cleaning Decisions Documented

### Close Approaches

| Column | Action | Reason |
|---|---|---|
| `close_approach_date` | Parsed to datetime | Required for temporal analysis |
| `velocity_infinity_km_s` | Imputed nulls with `velocity_km_s` | Physically equivalent at large distances |
| `absolute_magnitude` | Imputed nulls with column median | Median preserves central tendency; nulls are <2% |
| `distance_km`, `distance_lunar_distances` | Renamed from `dist_km`, `dist_lunar` | Descriptive consistency |
| `velocity_relative_km_h` | Renamed from `v_rel_kmh` | Descriptive consistency |

### NEA Catalogue

| Column | Action | Reason |
|---|---|---|
| `min_orbit_intersection_dist_au` | Dropped rows where null | MOID is critical hazard field; no imputation valid |
| `H → absolute_magnitude_h` | Renamed | JPL notation opaque for non-astronomers |
| `e → orbital_eccentricity` | Renamed | Standard orbital element, needs clarity |
| `a → semi_major_axis_au` | Renamed | Ditto |
| `i → orbital_inclination_deg` | Renamed | Ditto |
| `q → perihelion_dist_au` | Renamed | Ditto |
| `ad → aphelion_dist_au` | Renamed | Ditto |
| `per → orbital_period_days` | Renamed | Ditto |
| `n → mean_motion_deg_per_day` | Renamed | Ditto |
| `first_obs`, `last_obs` | Parsed to datetime | Needed for span calculation |

## 2.5 — Derived Columns Verification

In [ ]:
print('=== Close Approaches — Derived Columns ===')
derived_close = ['approach_year', 'approach_month', 'approach_month_name',
                 'approach_day_of_week', 'is_very_close_approach', 'speed_category']
for col in derived_close:
    if col in close_clean.columns:
        print(f'  ✓ {col}: {close_clean[col].dtype} — sample: {close_clean[col].iloc[0]}')

print('\n=== NEA Catalogue — Derived Columns ===')
derived_nea = ['is_named', 'observation_span_years', 'risk_tier', 'orbit_class_label']
for col in derived_nea:
    if col in nea_clean.columns:
        vc = nea_clean[col].value_counts().head(3).to_dict()
        print(f'  ✓ {col}: {nea_clean[col].dtype} — top values: {vc}')

## 2.6 — Save All 4 Processed Datasets

In [ ]:
saved = cleaning.save_cleaned_datasets(close_clean, nea_clean, PROCESSED_DIR)

print('\nSaved files:')
for name, path in saved.items():
    p = Path(path)
    size_mb = p.stat().st_size / 1_000_000
    print(f'  {p.name:<45} {size_mb:.1f} MB')

## 2.7 — Quick Validation of Processed Files

In [ ]:
datasets = {
    'nea_catalogue_clean.csv': 'NEA Catalogue (full)',
    'nea_hazardous_clean.csv': 'NEA — PHAs only',
    'close_approaches_clean.csv': 'Close Approaches (all)',
    'close_approaches_future_clean.csv': 'Close Approaches (future ≥2025)',
}

print(f"{'Dataset':<45} {'Rows':>8} {'Cols':>6}")
print('─' * 63)
for fname, label in datasets.items():
    fpath = PROCESSED_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath, nrows=0)  # header only
        n_rows = sum(1 for _ in open(fpath)) - 1  # fast count
        print(f'  {label:<43} {n_rows:>8,} {len(df.columns):>6}')
    else:
        print(f'  {label:<43}  MISSING')

## 2.8 — Summary

All four processed datasets are ready for EDA in notebook 03.

**Next step → Notebook 03: Exploratory Data Analysis**